# Hydro River Data Analysis: Arges, Vedea, Dambovita
## Extract and analyze hydro data from 2023-2026 using fuzzy matching for OCR-corrupted river names

## 1. Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
import os
import glob
import re
from fuzzywuzzy import fuzz, process
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Configuration
CSV_DIR = '/home/jovyan/hydro-ocr/csv_exports'
TARGET_RIVERS = ['Arges', 'Vedea'] #, 'Dambovita']
FUZZY_THRESHOLD = 80  # Minimum similarity score (0-100)

print(f"Working directory: {CSV_DIR}")
print(f"Target rivers: {TARGET_RIVERS}")
print(f"Fuzzy matching threshold: {FUZZY_THRESHOLD}")

Working directory: /home/jovyan/hydro-ocr/csv_exports
Target rivers: ['Arges', 'Vedea']
Fuzzy matching threshold: 80


## 2. Load and Explore CSV Files

In [2]:
# Get all CSV files
csv_files = sorted(glob.glob(os.path.join(CSV_DIR, 'rivers_part*.csv')))
print(f"Found {len(csv_files)} CSV files")
print(f"First file: {os.path.basename(csv_files[0])}")
print(f"Last file: {os.path.basename(csv_files[-1])}")

# Load first file to examine structure
df_sample = pd.read_csv(csv_files[0])
print(f"\nDataFrame shape: {df_sample.shape}")
print(f"\nColumn names:\n{df_sample.columns.tolist()}")
print(f"\nData types:\n{df_sample.dtypes}")
print(f"\nFirst few rows:")
df_sample.head()

Found 245 CSV files
First file: rivers_part0001.csv
Last file: rivers_part0245.csv

DataFrame shape: (200, 21)

Column names:
['date', 'bulletin_url', 'image_source', 'generation_key', 'image_key', 'nr_crt', 'raul', 'statia_hidrometrica', 'jud', 'CA_cm', 'CI_cm', 'CP_cm', 'Qmed_apr_m3_s', 'diag_H_cm', 'diag_Q_m3_s', 'diag_dH_cm', 'diag_GP', 'prog_H_cm', 'prog_Q_m3_s', 'prog_dH_cm', 'prog_GP']

Data types:
date                       str
bulletin_url               str
image_source               str
generation_key             str
image_key                  str
nr_crt                   int64
raul                       str
statia_hidrometrica        str
jud                        str
CA_cm                    int64
CI_cm                    int64
CP_cm                      str
Qmed_apr_m3_s              str
diag_H_cm                  str
diag_Q_m3_s                str
diag_dH_cm                 str
diag_GP                    str
prog_H_cm                  str
prog_Q_m3_s                str
pr

,date,bulletin_url,image_source,generation_key,image_key,nr_crt,raul,statia_hidrometrica,jud,CA_cm,...,CP_cm,Qmed_apr_m3_s,diag_H_cm,diag_Q_m3_s,diag_dH_cm,diag_GP,prog_H_cm,prog_Q_m3_s,prog_dH_cm,prog_GP
0,2023-01-01,https://www.hidro.ro/bulletin/prognoza-hidrolo...,https://www.hidro.ro/wp-content/uploads/2023/0...,pg2_2-2__f8883c32adbd12f3__grounding__eos__00a...,pg2_2-2__f8883c32adbd12f3,1,Vișeu,Bistra,220,300,...,"17,8",97,"53,0",NaN,NaN,NaN,94,"49,6",NaN,NaN
1,2023-01-01,https://www.hidro.ro/bulletin/prognoza-hidrolo...,https://www.hidro.ro/wp-content/uploads/2023/0...,pg2_2-2__f8883c32adbd12f3__grounding__eos__00a...,pg2_2-2__f8883c32adbd12f3,2,Iza,Vadu Izei,300,390,...,"12,7",130,"16,1",NaN,NaN,NaN,120,"12,4",NaN,NaN
2,2023-01-01,https://www.hidro.ro/bulletin/prognoza-hidrolo...,https://www.hidro.ro/wp-content/uploads/2023/0...,pg2_2-2__f8883c32adbd12f3__grounding__eos__00a...,pg2_2-2__f8883c32adbd12f3,3,Tur,Turulung,360,420,...,"10,8",285,"18,4",NaN,NaN,NaN,240,"16,8",NaN,NaN
3,2023-01-01,https://www.hidro.ro/bulletin/prognoza-hidrolo...,https://www.hidro.ro/wp-content/uploads/2023/0...,pg2_2-2__f8883c32adbd12f3__grounding__eos__00a...,pg2_2-2__f8883c32adbd12f3,4,Someșul Mare,Belecan,180,250,...,"33,1",74,"95,4",NaN,NaN,NaN,69,"86,7",NaN,NaN
4,2023-01-01,https://www.hidro.ro/bulletin/prognoza-hidrolo...,https://www.hidro.ro/wp-content/uploads/2023/0...,pg2_2-2__f8883c32adbd12f3__grounding__eos__00a...,pg2_2-2__f8883c32adbd12f3,5,Someș,Dej,450,550,...,"51,0",220,118,NaN,NaN,NaN,210,102,NaN,NaN


## 3. Normalize and Clean River Names

In [3]:
def normalize_river_name(name):
    """
    Normalize river names by:
    - Converting to lowercase
    - Removing extra spaces
    - Removing special characters (é, ș, ț, etc. -> e, s, t)
    - Handling common OCR errors
    """
    if not isinstance(name, str):
        return ""
    
    # Convert to lowercase
    name = name.lower().strip()
    
    # Remove common OCR artifacts and accents
    replacements = {
        'é': 'e', 'è': 'e', 'ê': 'e', 'ë': 'e',
        'á': 'a', 'à': 'a', 'â': 'a', 'ä': 'a', 'ă': 'a',
        'í': 'i', 'ì': 'i', 'î': 'i', 'ï': 'i',
        'ó': 'o', 'ò': 'o', 'ô': 'o', 'ö': 'o',
        'ú': 'u', 'ù': 'u', 'û': 'u', 'ü': 'u',
        'ș': 's', 'ş': 's',
        'ț': 't', 'ţ': 't',
        'ć': 'c', 'č': 'c', 'ç': 'c',
        'ń': 'n', 'ň': 'n',
        'ý': 'y', 'ÿ': 'y',
        'ů': 'u',
        'đ': 'd',
        'ø': 'o',
        'æ': 'ae',
        'œ': 'oe',
    }
    
    for old, new in replacements.items():
        name = name.replace(old, new)
    
    # Remove extra spaces
    name = ' '.join(name.split())
    
    return name

# Test normalization
print("Testing normalization:")
test_names = ['Argeș', 'Argees', 'Arges', 'ARG ES', 'Vedea', 'Vedéa', 'Dâmbovita', 'Dambovita', 'dAMBOVITA']
for test_name in test_names:
    normalized = normalize_river_name(test_name)
    print(f"  '{test_name}' -> '{normalized}'")

Testing normalization:
  'Argeș' -> 'arges'
  'Argees' -> 'argees'
  'Arges' -> 'arges'
  'ARG ES' -> 'arg es'
  'Vedea' -> 'vedea'
  'Vedéa' -> 'vedea'
  'Dâmbovita' -> 'dambovita'
  'Dambovita' -> 'dambovita'
  'dAMBOVITA' -> 'dambovita'


## 4. Extract Target Rivers Using Fuzzy Matching

In [4]:
def fuzzy_match_river(river_name, targets, threshold=FUZZY_THRESHOLD):
    """
    Use fuzzy matching to find the best matching target river.
    Returns the matched river name or None if no good match found.
    """
    if not isinstance(river_name, str) or not river_name.strip():
        return None
    
    normalized = normalize_river_name(river_name)
    if not normalized:
        return None
    
    # Use token_set_ratio for better matching with similar but not identical strings
    best_match = process.extractOne(normalized, targets, scorer=fuzz.token_set_ratio)
    
    if best_match and best_match[1] >= threshold:
        return best_match[0]
    
    return None

# Test fuzzy matching
print("Testing fuzzy matching:")
test_rivers = ['Argeș', 'Argees', 'Arges Malu', 'Vedea', 'Vedéa River', 'Dâmbovita', 'Dambovita SRL',
               'Olt', 'Siret', 'Buzau', 'Andere']
normalized_targets = [normalize_river_name(t) for t in TARGET_RIVERS]

for test_river in test_rivers:
    match = fuzzy_match_river(test_river, normalized_targets)
    print(f"  '{test_river}' -> {match}")

Testing fuzzy matching:
  'Argeș' -> arges
  'Argees' -> arges
  'Arges Malu' -> arges
  'Vedea' -> vedea
  'Vedéa River' -> vedea
  'Dâmbovita' -> None
  'Dambovita SRL' -> None
  'Olt' -> None
  'Siret' -> None
  'Buzau' -> None
  'Andere' -> None


In [22]:
# Load all CSV files and extract target rivers
print(f"Loading and processing {len(csv_files)} CSV files...")

column_set_jud = [
    'CA_cm', 'CI_cm', 'CP_cm', 'Qmed_apr_m3_s',
    'diag_H_cm', 'diag_Q_m3_s', 'diag_dH_cm', 'diag_GP',
    'prog_H_cm', 'prog_Q_m3_s', 'prog_dH_cm', 'prog_GP'
]

column_set_prog_H = [
    'prog_Q_m3_s', 'prog_dH_cm', 'prog_GP'
]

column_set_diag_GP = [
    'prog_H_cm', 'prog_Q_m3_s', 'prog_dH_cm', 'prog_GP'
]

column_set_prog_Q = [
    'prog_dH_cm', 'prog_GP'
]

def shift_numeric_jud_block(frame: pd.DataFrame, first_col: str, column_set: list[str], mask, shift_forwards: bool = True, clear_first_col: bool = False):
    if first_col not in frame.columns:
        return frame

    first_col_text = frame[first_col].fillna('').astype(str).str.strip()
    first_col_mask = mask(first_col_text)

    if first_col_mask.any():
        affected = int(first_col_mask.sum())
        print(f"  Found {affected} rows with numeric {first_col} values; shifting measurement block {('right' if shift_forwards else 'left')} and clearing {first_col}")

        column_set_test = column_set.copy()
        column_set_test.insert(0, first_col)  # Ensure the first column is included for shifting
        shifted = frame.loc[first_col_mask, column_set_test].copy()

        if shift_forwards:
            for target_col, source_col in zip(column_set_test[1:], column_set_test[:-1]):
                frame.loc[first_col_mask, target_col] = shifted[source_col].values
        else:
            for target_col, source_col in zip(column_set_test[:-1], column_set_test[1:]):
                # print(f"    Shifting {source_col} -> {target_col}")
                frame.loc[first_col_mask, target_col] = shifted[source_col].values

        if clear_first_col:
            frame.loc[first_col_mask, first_col] = ''

    return frame


all_data = []
normalized_targets = [normalize_river_name(t) for t in TARGET_RIVERS]

for idx, csv_file in enumerate(csv_files):
    try:
        df = pd.read_csv(csv_file, dtype=str, keep_default_na=False, na_filter=False).rename(
            columns={"diag_G": "diag_GP", "prog_G": "prog_GP"}
        )

        if 'raul' in df.columns and 'date' in df.columns:
            # Keep the row structure consistent before matching rivers.
            df = shift_numeric_jud_block(df, 'jud', column_set_jud, mask=lambda x: x.str.fullmatch(r'\d+(?:[.,]\d+)?').fillna(False), shift_forwards=True, clear_first_col=True)

            # Also shift the even more right prog_H block if it looks like prog_H is numeric
            df = shift_numeric_jud_block(df, 'prog_H_cm', column_set_prog_H, mask=lambda x: x == "", shift_forwards=False)

            # Also shift if diag_GP is numeric
            df = shift_numeric_jud_block(df, 'diag_GP', column_set_diag_GP, mask=lambda x: x.str.fullmatch(r'\-?\d+(?:[.,]\d+)?').fillna(False), shift_forwards=True, clear_first_col=True)

            # Shift prog_Q_m3_s if it is a letter from endng up to it
            df = shift_numeric_jud_block(df, 'prog_Q_m3_s', column_set_prog_Q, mask=lambda x: x.str.match(r'^[A-Za-z]+$').fillna(False), shift_forwards=False)            

            # Shift prog_H_cm if it is a letter from endng up to it. Completes the case where both it and prog_Q_m3_s are letters
            df = shift_numeric_jud_block(df, 'prog_H_cm', column_set_prog_H, mask=lambda x: x.str.match(r'^[A-Za-z]+$').fillna(False), shift_forwards=False)

            df = shift_numeric_jud_block(df, 'prog_H_cm', ["prog_Q_m3_s",	"prog_dH_cm"], mask=lambda x: x.str.fullmatch(r'\-?\d+(?:[.,]\d+)+').fillna(False), shift_forwards=False,)

            # If diag_dH_cm is a letter, make diag_GP take its value and clear it
            # Also clear diag_GP if it is numeric (artifact of shifting)
            diag_DH_mask = df['diag_dH_cm'].str.match(r'^[A-Za-z]+$').fillna(False)
            df.loc[diag_DH_mask, 'diag_GP'] = df.loc[diag_DH_mask, 'diag_dH_cm']
            df.loc[diag_DH_mask, 'diag_dH_cm'] = ''
            diag_GP_numeric_mask = df['diag_GP'].str.fullmatch(r'\-?\d+(?:[.,]\d+)?').fillna(False)
            df.loc[diag_GP_numeric_mask, 'diag_GP'] = ''

            # Same for prog_dH_cm and prog_GP
            # Also clear prog_GP if it is numeric (artifact of shifting)
            prog_DH_mask = df['prog_dH_cm'].str.match(r'^[A-Za-z]+$').fillna(False)
            df.loc[prog_DH_mask, 'prog_GP'] = df.loc[prog_DH_mask, 'prog_dH_cm']
            df.loc[prog_DH_mask, 'prog_dH_cm'] = ''
            prog_DH_identical_mask = df['prog_dH_cm'].str.strip() == df['prog_Q_m3_s'].str.strip()
            df.loc[prog_DH_identical_mask, 'prog_dH_cm'] = ''
            prog_GP_numeric_mask = df['prog_GP'].str.fullmatch(r'\-?\d+(?:[.,]\d+)?').fillna(False)
            df.loc[prog_GP_numeric_mask, 'prog_GP'] = ''

            df['raul_matched'] = df['raul'].apply(
                lambda x: fuzzy_match_river(x, normalized_targets, FUZZY_THRESHOLD)
            )

            df_target = df[df['raul_matched'].notna()].copy()

            # Drop the non-required "Arieș" river
            df_target = df_target[~df_target['raul'].str.lower().isin(['arieș', "arieş", "aries"])].copy()

            # Drop columns from differnt format or wrongly introduced by OCR errors
            columns_to_filter = [col for col in df_target.columns if re.match(r"col_.*|\d{2}\.\d{2}\.\d{4}", col)]
            df_target = df_target.drop(columns=columns_to_filter)

            # Drop leftover columns
            df_target = df_target.drop(columns=["Stação hidrométrica","Râul"], errors='ignore')

            if len(df_target) > 0:
                all_data.append(df_target)

        if (idx + 1) % 50 == 0:
            print(f"  Processed {idx + 1}/{len(csv_files)} files")

    except Exception as e:
        print(f"  Error reading {os.path.basename(csv_file)}: {e}")

# Combine all data
if all_data:
    df_extracted = pd.concat(all_data, ignore_index=True)
    # Fix date issues
    df_extracted.loc[df_extracted["image_source"] == "https://www.hidro.ro/wp-content/uploads/2025/08/3-19.jpg", "date"] = "2025-08-26"
    df_extracted.loc[df_extracted["date"] == "2023-02-29", "date"] = "2023-03-01"
    print(f"\nExtracted {len(df_extracted)} records for target rivers")
    print(f"Date range: {df_extracted['date'].min()} to {df_extracted['date'].max()}")
else:
    print("No data found for target rivers!")
    df_extracted = pd.DataFrame()

# Punctual fixes
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-02-09", df_extracted["raul_matched"] == "vedea"), ["prog_H_cm", "prog_Q_m3_s", "prog_dH_cm", "prog_GP"]] = ("138", "5,63", "", "")
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-02-09", df_extracted["raul_matched"] == "arges"), ["prog_H_cm", "prog_Q_m3_s", "prog_dH_cm", "prog_GP"]] = ("-160", "30,0", "", "")

df_extracted.loc[np.logical_and(df_extracted["date"] == "2024-02-01", df_extracted["raul_matched"] == "vedea"), ["prog_H_cm", "prog_Q_m3_s", "prog_dH_cm", "prog_GP"]] = ("116", "2,14", "", "")
df_extracted.loc[np.logical_and(df_extracted["date"] == "2024-02-01", df_extracted["raul_matched"] == "arges"), ["prog_H_cm", "prog_Q_m3_s", "prog_dH_cm", "prog_GP"]] = ("-188", "15,3", "", "")

df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-02-11", df_extracted["raul_matched"] == "vedea"), ["diag_GP", "prog_H_cm", "prog_Q_m3_s", "prog_dH_cm", "prog_GP"]] = ("G", "135", "5,08", "", "")
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-02-11", df_extracted["raul_matched"] == "arges"), ["prog_H_cm", "prog_Q_m3_s", "prog_dH_cm", "prog_GP"]] = ("-163", "28,2", "", "")

df_extracted.loc[np.logical_and(df_extracted["date"] == "2025-01-17", df_extracted["raul_matched"] == "vedea"), ["prog_H_cm", "prog_Q_m3_s", "prog_dH_cm", "prog_GP"]] = ("118", "1,54", "", "")
df_extracted.loc[np.logical_and(df_extracted["date"] == "2025-01-17", df_extracted["raul_matched"] == "arges"), ["prog_H_cm", "prog_Q_m3_s", "prog_dH_cm", "prog_GP"]] = ("106", "17,3", "", "")

df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-06-23", df_extracted["raul_matched"] == "vedea"), ["diag_dH_cm", "prog_H_cm", "prog_Q_m3_s", "prog_dH_cm", "prog_GP"]] = ("", "120", "2,64", "", "")

df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-05-21", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s"]] = ("380", "500", "550", "7,95", "122", "2,94",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-05-21", df_extracted["raul_matched"] == "arges"), ["Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s"]] = ("82,0", "-123", "59,5",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-05-22", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s"]] = ("380", "500", "550", "7,95", "122", "2,94",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-06-23", df_extracted["raul_matched"] == "arges"), ["diag_dH_cm", "diag_GP", "prog_H_cm", "prog_Q_m3_s"]] = ("", "", "-77", "108",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2025-12-14", df_extracted["raul_matched"] == "arges"), ["prog_H_cm", "prog_Q_m3_s"]] = ("156", "42,2",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2025-12-14", df_extracted["raul_matched"] == "vedea"), ["prog_H_cm", "prog_Q_m3_s"]] = ("128", "4,40",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2026-02-22", df_extracted["raul_matched"] == "vedea"), ["prog_H_cm"]] = ("1089",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2025-02-27", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "14,5", "118", "1,54", "118", "1,54",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2025-02-27", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("510", "540", "600", "24,1", "107", "15,8", "106", "15,6",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-09-05", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "2,38", "113", "1,77", "113", "1,77",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-09-05", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("210", "240", "300", "22,0", "-180", "18,5", "-182", "16,7",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-10-16", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "5,89", "114", "1,90", "114", "1,90",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-10-16", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("210", "240", "300", "22,8", "-179", "18,9", "-180", "18,5",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-09-17", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "2,38", "114", "1,90", "115", "2,02",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-09-17", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("210", "240", "300", "22,0", "-175", "20,5", "-171", "22,1",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-09-18", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "2,38", "114", "1,90", "114", "1,90",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-09-18", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("210", "240", "300", "22,0", "-176", "20,1", "-176", "20,1",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-09-21", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "2,38", "114", "1,90", "114", "1,90",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-09-21", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("210", "240", "300", "22,0", "-181", "18,1", "-181", "18,1",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-10-21", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "5,89", "113", "1,77", "113", "1,77",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-10-21", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("210", "240", "300", "22,8", "-180", "18,5", "-180", "18,5",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-10-26", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "5,89", "113", "1,77", "113", "1,77",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-10-26", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("210", "240", "300", "22,8", "-183", "17,3", "-182", "17,7",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-09-27", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "2,38", "114", "1,90", "114", "1,90",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-09-27", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("210", "240", "300", "22,0", "-173", "21,3", "-174", "20,9",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-09-28", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "2,38", "114", "1,90", "114", "1,90",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-09-28", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("210", "240", "300", "22,0", "-175", "20,5", "-176", "20,1",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-10-28", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "5,89", "113", "1,77", "113", "1,77",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-10-28", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("210", "240", "300", "22,8", "-182", "17,7", "-182", "17,7",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-09-29", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "2,38", "112", "1,65", "112", "1,65",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-09-29", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("210", "240", "300", "22,0", "-181", "18,1", "-182", "17,7",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-10-29", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "5,89", "113", "1,77", "113", "1,77",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-10-29", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("210", "240", "300", "22,8", "-183", "17,3", "-183", "17,3",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-09-30", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "2,38", "110", "1,40", "110", "1,40",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-09-30", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("210", "240", "300", "22,0", "-179", "18,9", "-181", "18,1",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2025-07-27", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "6,16", "115", "0,84", "115", "0,84",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2025-07-27", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("510", "540", "600", "44,8", "121", "20,4", "119", "19,7",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2025-07-28", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "6,16", "115", "0,84", "115", "0,84",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2025-07-28", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("510", "540", "600", "44,8", "119", "19,6", "119", "19,6",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2025-07-29", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "6,16", "115", "0,84", "115", "0,84",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2025-07-29", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("510", "540", "600", "44,8", "116", "18,6", "119", "19,6",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2025-07-30", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "6,16", "115", "0,84", "115", "0,84",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2025-07-30", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("510", "540", "600", "44,8", "121", "20,4", "123", "21,2",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2025-07-31", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "6,16", "115", "0,84", "115", "0,84",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2025-07-31", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("510", "540", "600", "44,8", "119", "19,7", "116", "18,6",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-12-10", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "5,75", "115", "2,02", "115", "2,02",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-12-10", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("210", "240", "300", "25,8", "-182", "17,7", "-185", "16,5",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-12-18", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "5,75", "114", "1,90", "114", "1,90",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-12-18", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("210", "240", "300", "25,8", "-189", "14,9", "-190", "14,5",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-12-28", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "5,75", "115", "2,02", "115", "2,02",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-12-28", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("210", "240", "300", "25,8", "-189", "14,9", "-190", "14,5",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-12-29", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "5,75", "115", "2,02", "115", "2,02",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-12-29", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("210", "240", "300", "25,8", "-183", "17,3", "-185", "16,5",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2024-10-27", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "5,89", "113", "0,664", "113", "0,664",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2024-10-27", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("510", "540", "600", "22,8", "107", "17,6", "105", "17,0",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2024-10-28", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "5,89", "113", "0,664", "113", "0,664",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2024-10-28", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("510", "540", "600", "22,8", "110", "18,4", "109", "18,2",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2024-10-29", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "5,89", "113", "0,664", "113", "0,664",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2024-10-29", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("510", "540", "600", "22,8", "110", "18,4", "109", "18,2",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2024-10-30", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "5,89", "113", "0,664", "113", "0,664",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2024-10-30", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("510", "540", "600", "22,8", "108", "17,8", "107", "17,4",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2026-03-28", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "20,6", "106", "5,30", "108", "5,60",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2026-03-28", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("510", "540", "600", "35,3", "130", "25,0", "136", "28,6",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2026-03-29", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "20,6", "106", "5,30", "109", "0,360",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2026-03-29", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("510", "540", "600", "35,3", "165", "49,0", "190", "71,0",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2026-03-30", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "20,6", "150", "14,1", "160", "16,8",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2026-03-30", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("510", "540", "600", "35,3", "158", "43,6", "158", "43,6",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2026-03-31", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "20,6", "138", "11,1", "138", "11,1",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2026-03-31", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("510", "540", "600", "35,3", "139", "30,4", "139", "30,4",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2025-12-28", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "5,75", "122", "2,60", "122", "2,60",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2025-12-28", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("510", "540", "600", "25,8", "162", "46,6", "172", "54,8",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2025-12-29", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "5,75", "122", "2,60", "122", "2,60",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2025-12-29", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("510", "540", "600", "25,8", "157", "42,9", "154", "40,8",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2025-12-30", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "5,75", "122", "2,60", "122", "2,60",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2025-12-30", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("510", "540", "600", "25,8", "126", "23,0", "133", "26,8",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2025-12-31", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "5,75", "122", "2,60", "122", "2,60",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2025-12-31", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("510", "540", "600", "25,8", "122", "21,0", "120", "20,0",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-08-02", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "2,83", "114", "1,90", "114", "1,90",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-08-02", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("210", "240", "300", "29,5", "-158", "30,5", "-162", "27,7",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-08-10", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "2,83", "113", "1,77", "113", "1,77",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-08-10", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("210", "240", "300", "29,5", "-169", "23,1", "-170", "22,5",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-08-25", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "2,83", "114", "1,90", "114", "1,90",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-08-25", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("210", "240", "300", "29,5", "-177", "19,7", "-179", "18,9",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-08-28", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "2,83", "113", "1,77", "113", "1,77",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-08-28", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("210", "240", "300", "29,5", "-180", "18,5", "-184", "16,9",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2025-05-28", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "7,95", "138", "7,40", "140", "8,00",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2025-05-28", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("510", "540", "600", "82,0", "208", "95,2", "225", "119",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2025-05-29", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "7,95", "136", "6,80", "136", "6,80",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2025-05-29", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("510", "540", "600", "82,0", "227", "122", "225", "119",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2025-05-31", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "7,95", "135", "6,50", "135", "6,50",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2025-05-31", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("510", "540", "600", "82,0", "224", "118", "223", "117",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2024-12-28", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "5,75", "126", "3,80", "123", "2,90",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2024-12-28", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("510", "540", "600", "25,8", "113", "19,2", "116", "20,1",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2024-12-30", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "5,75", "126", "3,80", "126", "3,80",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2024-12-30", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("510", "540", "600", "25,8", "113", "19,2", "112", "18,9",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2024-12-31", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "5,75", "126", "3,80", "126", "3,80",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2024-12-31", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("510", "540", "600", "25,8", "112", "19,0", "111", "18,8",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2025-01-29", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "7,51", "116", "1,07", "116", "1,07",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2025-01-29", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("510", "540", "600", "22,0", "107", "17,6", "106", "17,4",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2025-01-30", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "7,51", "115", "0,840", "115", "0,840",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2025-01-30", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("510", "540", "600", "22,0", "107", "17,6", "106", "17,4",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2025-01-31", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "7,51", "115", "0,840", "115", "0,840",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2025-01-31", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("510", "540", "600", "22,0", "103", "16,4", "102", "16,2",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-11-13", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "3,89", "115", "2,02", "116", "2,14",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-11-13", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("210", "240", "300", "24,9", "-171", "22,1", "-177", "19,7",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-11-17", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "3,89", "115", "2,02", "116", "2,14",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-11-17", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("210", "240", "300", "24,9", "-181", "18,1", "-179", "18,9",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-11-21", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "3,89", "115", "2,02", "115", "2,02",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-11-21", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("210", "240", "300", "24,9", "-178", "19,3", "-179", "18,9",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-11-28", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "3,89", "120", "2,64", "119", "2,52",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-11-28", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("210", "240", "300", "24,9", "-188", "15,3", "-183", "17,3",)

df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-09-01", df_extracted["raul_matched"] == "arges"), ["diag_H_cm"]] = ("-185",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-12-05", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "5,75", "117", "2,27", "117", "2,27",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-11-09", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("210", "240", "300", "24,9", "-184", "16,9", "-185", "16,5",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-12-19", df_extracted["raul_matched"] == "arges"), ["diag_H_cm"]] = ("-189",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2024-01-21", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "7,51", "116", "2,14", "116", "2,14",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-10-27", df_extracted["raul_matched"] == "arges"), ["diag_H_cm"]] = ("-182",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-11-27", df_extracted["raul_matched"] == "vedea"), ["prog_H_cm", "prog_Q_m3_s"]] = ("121", "2,79",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2022-12-28", df_extracted["raul_matched"] == "arges"), ["diag_H_cm", "diag_Q_m3_s"]] = ("-165", "27,0",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-03-30", df_extracted["raul_matched"] == "vedea"), ["prog_H_cm"]] = ("123",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-11-30", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "3,89", "120", "2,64", "120", "2,64",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-03-31", df_extracted["raul_matched"] == "vedea"), ["prog_H_cm"]] = ("123",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-08-31", df_extracted["raul_matched"] == "arges"), ["diag_H_cm"]] = ("-183",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-10-31", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "5,89", "113", "1,77", "113", "1,77",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-10-31", df_extracted["raul_matched"] == "arges"), ["diag_H_cm"]] = ("-182",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-12-31", df_extracted["raul_matched"] == "vedea"), ["prog_H_cm"]] = ("115",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2022-12-31", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("210", "240", "300", "25,8", "-165", "27,0", "-166", "26,4",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-08-06", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("210", "240", "300", "29,5", "-158", "30,5", "-148", "38,2",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-08-31", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "2,83", "113", "1,77", "114", "1,9",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-11-16", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("210", "240", "300", "24,9", "-183", "17,3", "-183", "17,3",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-12-19", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "5,75", "115", "2,02", "114", "1,9",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-11-26", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "3,89", "118", "2,39", "120", "2,64",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-10-27", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "5,89", "113", "1,77", "113", "1,77",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2024-02-29", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "14,5", "115", "2,02", "115", "2,02",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2024-02-29", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("510", "540", "600", "24,1", "120", "20,5", "120", "20,5",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2024-05-28", df_extracted["raul_matched"] == "vedea"), ["Qmed_apr_m3_s",]] = ("7,95", )
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-05-20", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "7,95", "122", "2,94", "122", "2,94",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-05-20", df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("510", "540", "600", "82,0", "-124", "58,6", "-124", "58,6",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-11-22", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "3,89", "115", "2,02", "116", "2,14",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2024-06-29", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "6,83", "116", "2,14", "116", "2,14",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2025-06-29", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "6,83", "122", "2,60", "121", "2,30",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-04-07", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "10,8", "152", "10,1", "147", "8,38",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-04-20", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "10,8", "129", "4,01", "129", "4,01",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2025-04-28", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "10,8", "114", "0,752", "114", "0,752",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-05-28", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "7,95", "122", "2,94", "124", "3,25",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-05-30", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "7,95", "123", "3,10", "123", "3,10",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-07-26", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "6,16", "114", "1,90", "114", "1,90",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-03-13", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "20,6", "123", "3,10", "123", "3,10",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-03-29", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "20,6", "123", "3,10", "123", "3,10",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2025-09-27", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "2,38", "117", "1,30", "117", "1,30",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-08-29", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "2,83", "113", "1,77", "114", "1,90",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2025-08-29", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "2,83", "121", "2,30", "120", "2,00",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2026-01-28", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "7,51", "132", "5,60", "215", "38,3",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-03-31", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "20,6", "123", "3,10", "123", "3,10",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2025-10-28", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "5,89", "123", "2,90", "123", "2,90",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2025-10-30", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "5,89", "123", "2,90", "123", "2,90",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-07-28", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "6,16", "114", "1,90", "114", "1,90",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2023-07-30", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "6,16", "114", "1,90", "114", "1,90",)
df_extracted.loc[np.logical_and(df_extracted["date"] == "2024-01-28", df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "7,51", "116", "2,14", "116", "2,14",)

# Find by image_key because date will later be changed
df_extracted.loc[np.logical_and(df_extracted["image_key"] == "3-5__25b4c4704072fa3c",df_extracted["raul_matched"] == "vedea"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("380", "500", "550", "6,83", "119", "2,52", "119", "2,52",)
df_extracted.loc[np.logical_and(df_extracted["image_key"] == "3-5__25b4c4704072fa3c",df_extracted["raul_matched"] == "arges"), ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]] = ("510", "540", "600", "68,8", "126", "24,1", "126", "24,1",)

# Remove anything with empty values in the columns of interest
# Most of them have been corrected, but a few have been left empty
for col in ["CA_cm", "CI_cm", "CP_cm", "Qmed_apr_m3_s", "diag_H_cm", "diag_Q_m3_s", "prog_H_cm", "prog_Q_m3_s"]:
    df_extracted = df_extracted[df_extracted[col] != ""].copy()


Loading and processing 245 CSV files...
  Found 150 rows with numeric jud values; shifting measurement block right and clearing jud
  Found 99 rows with numeric prog_H_cm values; shifting measurement block left and clearing prog_H_cm
  Found 49 rows with numeric diag_GP values; shifting measurement block right and clearing diag_GP
  Found 2 rows with numeric prog_H_cm values; shifting measurement block left and clearing prog_H_cm
  Found 149 rows with numeric jud values; shifting measurement block right and clearing jud
  Found 149 rows with numeric prog_H_cm values; shifting measurement block left and clearing prog_H_cm
  Found 11 rows with numeric diag_GP values; shifting measurement block right and clearing diag_GP
  Found 100 rows with numeric jud values; shifting measurement block right and clearing jud
  Found 100 rows with numeric prog_H_cm values; shifting measurement block left and clearing prog_H_cm
  Found 100 rows with numeric jud values; shifting measurement block right an

In [6]:
(pd.to_datetime(df_extracted.loc[df_extracted["prog_H_cm"] == "", "date"]) - pd.offsets.MonthBegin(1)).value_counts()

Series([], Name: count, dtype: int64)

In [25]:
df_extracted[np.logical_and.reduce([df_extracted["Qmed_apr_m3_s"] == "20,6", df_extracted["raul_matched"] == "vedea", df_extracted["image_source"].str.contains("/home")])]

,date,bulletin_url,image_source,generation_key,image_key,nr_crt,raul,statia_hidrometrica,jud,CA_cm,...,Qmed_apr_m3_s,diag_H_cm,diag_Q_m3_s,diag_dH_cm,diag_GP,prog_H_cm,prog_Q_m3_s,prog_dH_cm,prog_GP,raul_matched
1953,2023-03-13,https://www.hidro.ro/bulletin/prognoza-hidrolo...,/home/jovyan/hydro-ocr/deepseek_ocr_store/imag...,pg2_2-8-30969ccd6f__53c449574bff3530__free__eo...,pg2_2-8-30969ccd6f__53c449574bff3530,30,Vedea,Alexandria,,380,...,"20,6",123,"3,10",-158,,123,"3,10",,,vedea
1963,2023-03-17,https://www.hidro.ro/bulletin/prognoza-hidrolo...,/home/jovyan/hydro-ocr/deepseek_ocr_store/imag...,pg2_2-10-82e661470b__7082556c9089146f__free__e...,pg2_2-10-82e661470b__7082556c9089146f,30,Vedea,Alexandria,,380,...,"20,6",123,"3,10",,,123,"3,10",,,vedea
2059,2023-03-28,https://www.hidro.ro/bulletin/prognoza-hidrolo...,/home/jovyan/hydro-ocr/deepseek_ocr_store/imag...,pg2_2-15-6439d37376__ffdd4dfd8ee03195__free__e...,pg2_2-15-6439d37376__ffdd4dfd8ee03195,30,Vedea,Alexandria,,380,...,"20,6",123,"3,10",,,123,"3,10",,,vedea
2061,2025-03-28,https://www.hidro.ro/bulletin/prognoza-hidrolo...,/home/jovyan/hydro-ocr/deepseek_ocr_store/imag...,pg2_2-10-c411c27e58__1021d15d86ce0c9a__free__e...,pg2_2-10-c411c27e58__1021d15d86ce0c9a,30,Vedea,Alexandra,TR,380,...,"20,6",114,"0,752",,,114,"0,752",,,vedea
2063,2026-03-28,https://www.hidro.ro/bulletin/prognoza-hidrolo...,/home/jovyan/hydro-ocr/deepseek_ocr_store/imag...,3-22-e81624ca84__e6e5ebd54ddf3d0a__free__eos__...,3-22-e81624ca84__e6e5ebd54ddf3d0a,30,Vedea,Alexandra,GR,380,...,"20,6",106,"5,30",,,108,"5,60",,,vedea
2123,2023-03-29,https://www.hidro.ro/bulletin/prognoza-hidrolo...,/home/jovyan/hydro-ocr/deepseek_ocr_store/imag...,pg2_2-16-ccdcb1b405__0008ef42e8e2e2fb__free__e...,pg2_2-16-ccdcb1b405__0008ef42e8e2e2fb,30,Vedea,Alexandria,,380,...,"20,6",123,"3,10",,,123,"3,10",,,vedea
2127,2026-03-29,https://www.hidro.ro/bulletin/prognoza-hidrolo...,/home/jovyan/hydro-ocr/deepseek_ocr_store/imag...,3-23-550c473778__2db8f588199791dc__free__eos__...,3-23-550c473778__2db8f588199791dc,29,Vedea,Alexandra,TR,380,...,"20,6",106,"5,30",,,109,"0,360",,,vedea
2186,2023-03-30,https://www.hidro.ro/bulletin/prognoza-hidrolo...,/home/jovyan/hydro-ocr/deepseek_ocr_store/imag...,3-14-c04a0ac5a6__d3e44d68a6c85b84__free__eos__...,3-14-c04a0ac5a6__d3e44d68a6c85b84,30,Vedea,Alexandria,,380,...,"20,6",123,"3,10",,,123,"3,10",,,vedea
2190,2026-03-30,https://www.hidro.ro/bulletin/prognoza-hidrolo...,/home/jovyan/hydro-ocr/deepseek_ocr_store/imag...,3-24-7dde7819a7__f24b5306650bd769__free__eos__...,3-24-7dde7819a7__f24b5306650bd769,29,Vedea,Alexandra,TR,380,...,"20,6",150,"14,1",,,160,"16,8",,,vedea
2231,2023-03-31,https://www.hidro.ro/bulletin/prognoza-hidrolo...,/home/jovyan/hydro-ocr/deepseek_ocr_store/imag...,pg2_2-17-9f3ca59353__a8345c07a9c12253__free__e...,pg2_2-17-9f3ca59353__a8345c07a9c12253,30,Vedea,Alexandria,,380,...,"20,6",123,"3,10",,,123,"3,10",,,vedea


In [41]:
df_extracted[df_extracted["date"] == "2023-05-20"].values

array([['2023-05-20',
        'https://www.hidro.ro/bulletin/prognoza-hidrologica-pentru-rauri-in-intervalul-20-05-2023-ora-07-00-21-05-2023-ora-07-00/',
        'https://www.hidro.ro/wp-content/uploads/2023/05/3-18.jpg',
        '3-18__688807b86d5f8a5c__grounding__eos__00a105b332',
        '3-18__688807b86d5f8a5c', '30', 'Vedea', 'Alexandria', '', '380',
        '500', '550', '7,95', '122', '2,94', '', '', '122', '2,94', '',
        '', 'vedea'],
       ['2023-05-20',
        'https://www.hidro.ro/bulletin/prognoza-hidrologica-pentru-rauri-in-intervalul-20-05-2023-ora-07-00-21-05-2023-ora-07-00/',
        'https://www.hidro.ro/wp-content/uploads/2023/05/3-18.jpg',
        '3-18__688807b86d5f8a5c__grounding__eos__00a105b332',
        '3-18__688807b86d5f8a5c', '31', 'Argeș', 'Malu Spart', '', '510',
        '540', '600', '82,0', '-124', '58,6', '', '', '-124', '58,6', '',
        '', 'arges']], dtype=object)

In [26]:
df_extracted.assign(Qapr_m3_s=df_extracted["Qmed_apr_m3_s"].str.replace(",", ".").astype(float)).groupby("raul_matched")["Qapr_m3_s"].max()

raul_matched
arges    82.0
vedea    20.6
Name: Qapr_m3_s, dtype: float64

In [27]:
df_extracted[df_extracted["prog_H_cm"].str.contains(",")].drop(columns=["bulletin_url", "image_source", "generation_key", "image_key"])

,date,nr_crt,raul,statia_hidrometrica,jud,CA_cm,CI_cm,CP_cm,Qmed_apr_m3_s,diag_H_cm,diag_Q_m3_s,diag_dH_cm,diag_GP,prog_H_cm,prog_Q_m3_s,prog_dH_cm,prog_GP,raul_matched


In [28]:
problematic_mask = np.abs(df_extracted["prog_H_cm"].astype("int") - df_extracted["diag_H_cm"].astype("int")) > 50

In [29]:
np.logical_and(df_extracted["date"] < "2023-05-01", np.abs(df_extracted["prog_H_cm"].astype("int") - df_extracted["diag_H_cm"].astype("int")) > 100).sum()

np.int64(0)

In [30]:
df_extracted[problematic_mask].drop(columns=["bulletin_url", "generation_key", "image_key"])#.values

,date,image_source,nr_crt,raul,statia_hidrometrica,jud,CA_cm,CI_cm,CP_cm,Qmed_apr_m3_s,diag_H_cm,diag_Q_m3_s,diag_dH_cm,diag_GP,prog_H_cm,prog_Q_m3_s,prog_dH_cm,prog_GP,raul_matched
68,2025-12-01,https://www.hidro.ro/wp-content/uploads/2025/1...,30,Vedea,Alexandria,TR,380,500,550,"5,75",125,"3,50",,,185,"25,7",,,vedea
146,2025-12-02,https://www.hidro.ro/wp-content/uploads/2025/1...,30,Vedea,Alexandria,TR,380,500,550,"5,75",125,"3,50",,,185,"25,7",,,vedea
448,2026-02-07,https://www.hidro.ro/wp-content/uploads/2026/0...,30,Vedea,Alexandria,TR,380,500,550,"14,5",140,"8,00",,,220,"40,4",,,vedea
512,2026-01-08,https://www.hidro.ro/wp-content/uploads/2026/0...,30,Vedea,Alexandria,TR,380,500,550,"7,51",122,"2,60",,,200,"32,0",,,vedea
1391,2026-02-20,https://www.hidro.ro/wp-content/uploads/2026/0...,30,Vedea,Alexandria,TR,380,500,550,"14,5",165,"17,3",,,295,"75,2",,,vedea
1523,2026-02-22,https://www.hidro.ro/wp-content/uploads/2026/0...,30,Vedea,Alexandria,TR,380,500,550,"14,5",220,"40,4",,,1089,"35,8",539 CB,,vedea
1798,2025-12-25,https://www.hidro.ro/wp-content/uploads/2025/1...,31,Argeș,Malu Spart,GR,510,540,600,"25,8",456,585,,,146,"35,2",,,arges
2053,2026-01-28,/home/jovyan/hydro-ocr/deepseek_ocr_store/imag...,30,Vedea,Alexandra,TR,380,500,550,"7,51",132,"5,60",,,215,"38,3",,,vedea


In [ ]:
cols_to_verify = ["CA_cm",	"CI_cm",	"CP_cm",	"Qmed_apr_m3_s",	"diag_H_cm",	"diag_Q_m3_s",	"diag_dH_cm",	"diag_GP",	"prog_H_cm",	"prog_Q_m3_s",	"prog_dH_cm",	"prog_GP"]
df_extracted_to_check = df_extracted[cols_to_verify + ["date", "raul", "image_source"]].replace("", np.nan).replace("NIL", np.nan).replace("lipsă", np.nan)
# df_extracted_to_check.assign(**{col: df_extracted_to_check[col].str.replace(",", ".").astype("Float64") for col in cols_to_verify})

In [12]:
def try_convert_to_numeric(text: str):
    try:
        return float(text.replace(",", "."))
    except Exception as e:
        return np.nan

df_extracted_to_check["prog_GP"].value_counts().index[df_extracted_to_check["prog_GP"].value_counts().index.map(try_convert_to_numeric).isna()]

Index(['N', 'G', 'P', 'GN'], dtype='str', name='prog_GP')

# 5. Remove duplicates

In [31]:
df_extracted.loc[df_extracted["image_key"] == "3-5__657553304c0e3d3d", "date"] = "2023-03-12"
df_extracted.loc[df_extracted["image_key"] == "3__06117b95776aa791", "date"] = "2024-06-01"
df_extracted.loc[df_extracted["image_key"] == "3-1__34aad94a962d1bc8", "date"] = "2024-06-02"
df_extracted.loc[df_extracted["image_key"] == "3-2__e7394eead42567cb", "date"] = "2024-06-03"
df_extracted.loc[df_extracted["image_key"] == "3-3__3fbd582f852d55de", "date"] = "2024-06-04"
df_extracted.loc[df_extracted["image_key"] == "3-4__c7fa5085e7fe8a7f", "date"] = "2024-06-05"
df_extracted.loc[df_extracted["image_key"] == "3-5__25b4c4704072fa3c", "date"] = "2024-06-06"


dates_to_shift = pd.date_range(start="2024-09-10", end="2024-09-20")[::-1]
if not np.isin(dates_to_shift, pd.to_datetime(df_extracted["date"])).all():
    for initial_date, next_date in zip(dates_to_shift, dates_to_shift + pd.Timedelta(days=1)):
        df_extracted.loc[df_extracted["date"] == initial_date.strftime("%Y-%m-%d"), "date"] = next_date.strftime("%Y-%m-%d")

df_extracted.loc[df_extracted["image_key"] == "3-9__521989937c555068", "date"] = "2024-09-10"
df_extracted.loc[df_extracted["image_key"] == "3-13__a5a1aff1339c4218", "date"] = "2025-01-22"

# Remaining duplicates are due to multiple entries for the same date and raul, so we keep only one
df_extracted = df_extracted.drop_duplicates(subset=["date", "raul_matched", "image_key"]).copy()

# There are still a few cases where only one raul per image was decoded, so we remove the entirely
df_extracted_per_river = df_extracted.groupby("date").apply(lambda x: x.shape[0]) 
df_extracted = df_extracted[df_extracted["date"].isin(df_extracted_per_river[df_extracted_per_river == len(TARGET_RIVERS)].index)].copy()

# Station names are parsed wrong on accasion, hardcoding them
df_extracted.loc[df_extracted["raul_matched"] == "arges", "statia_hidrometrica"] = "Malu Spart"
df_extracted.loc[df_extracted["raul_matched"] == "vedea", "statia_hidrometrica"] = "Alexandria"

In [32]:
dates_to_shift[~np.isin(dates_to_shift, pd.to_datetime(df_extracted["date"]))]

DatetimeIndex(['2024-09-18'], dtype='datetime64[us]', freq='-1D')

In [33]:
duplicated_lines = df_extracted[df_extracted.duplicated(subset=["date", "raul"], keep=False)].sort_values(by=["date", "image_key", "raul"]).values

In [34]:
duplicated_lines

array([], shape=(0, 22), dtype=object)

In [35]:
df_extracted[['raul_matched', 'statia_hidrometrica']].value_counts()

raul_matched  statia_hidrometrica
vedea         Alexandria             1089
arges         Malu Spart             1089
Name: count, dtype: int64

In [36]:
df_extracted[['raul', 'statia_hidrometrica']].value_counts()

raul   statia_hidrometrica
Vedea  Alexandria             1089
Argeș  Malu Spart             1073
Argeş  Malu Spart               10
Arges  Malu Spart                4
Arge?  Malu Spart                2
Name: count, dtype: int64

## 5. Combine and Sort Data by Date

In [37]:
# Convert date column to datetime and sort
df_extracted['date'] = pd.to_datetime(df_extracted['date'], errors='coerce')
df_extracted = df_extracted.dropna(subset=['date'])
df_extracted = df_extracted.sort_values('date').reset_index(drop=True)

print(f"Total records after sorting: {len(df_extracted)}")
print(f"Date range: {df_extracted['date'].min().date()} to {df_extracted['date'].max().date()}")

# Display data distribution by river
print("\nRecords by river:")
river_counts = df_extracted['raul_matched'].value_counts()
print(river_counts)

# Show sample data
print("\nSample of extracted data:")
df_extracted[['date', 'raul', 'raul_matched', 'statia_hidrometrica', 'Qmed_apr_m3_s']].head(10)

Total records after sorting: 2178
Date range: 2022-12-28 to 2026-04-19

Records by river:
raul_matched
arges    1089
vedea    1089
Name: count, dtype: int64

Sample of extracted data:


,date,raul,raul_matched,statia_hidrometrica,Qmed_apr_m3_s
0,2022-12-28,Argeș,arges,Malu Spart,"25,8"
1,2022-12-28,Vedea,vedea,Alexandria,"16,7"
2,2022-12-30,Argeș,arges,Malu Spart,"25,8"
3,2022-12-30,Vedea,vedea,Alexandria,"16,7"
4,2022-12-31,Argeș,arges,Malu Spart,"25,8"
5,2022-12-31,Vedea,vedea,Alexandria,"16,7"
6,2023-01-01,Vedea,vedea,Alexandria,"7,51"
7,2023-01-01,Argeș,arges,Malu Spart,"22,0"
8,2023-01-02,Argeș,arges,Malu Spart,"22,0"
9,2023-01-02,Vedea,vedea,Alexandria,"7,51"


## 6. Calculate Missing Days

In [38]:
# Get date range
if len(df_extracted) > 0:
    first_date = df_extracted['date'].min()
    last_date = df_extracted['date'].max()
    
    # Generate complete date sequence
    date_range = pd.date_range(start=first_date, end=last_date, freq='D')
    complete_dates = set(date_range.date)
    
    # Get actual dates in dataset
    actual_dates = set(df_extracted['date'].dt.date)
    
    # Calculate missing dates
    missing_dates = sorted(complete_dates - actual_dates)
    
    # Statistics
    total_days = len(date_range)
    present_days = len(actual_dates)
    missing_days_count = len(missing_dates)
    coverage_percent = (present_days / total_days) * 100 if total_days > 0 else 0
    
    print(f"Date Analysis:")
    print(f"  First date: {first_date.date()}")
    print(f"  Last date: {last_date.date()}")
    print(f"  Total span (days): {total_days}")
    print(f"  Days with data: {present_days}")
    print(f"  Missing days: {missing_days_count}")
    print(f"  Coverage: {coverage_percent:.2f}%")
    
    # Show first and last missing dates
    if missing_dates:
        print(f"\n  First missing date: {missing_dates[0]}")
        print(f"  Last missing date: {missing_dates[-1]}")
    
    # Show gaps in data (consecutive missing days)
    print(f"\nGap Analysis (consecutive missing days):")
    if missing_dates:
        gaps = []
        gap_start = missing_dates[0]
        gap_end = missing_dates[0]
        
        for i in range(1, len(missing_dates)):
            if (missing_dates[i] - missing_dates[i-1]).days == 1:
                gap_end = missing_dates[i]
            else:
                gaps.append((gap_start, gap_end))
                gap_start = missing_dates[i]
                gap_end = missing_dates[i]
        gaps.append((gap_start, gap_end))
        
        print(f"  Number of gaps: {len(gaps)}")
        print(f"  Largest gap: {max((end - start).days + 1 for start, end in gaps)} days")
        
        # Show top 10 largest gaps
        gaps_sorted = sorted(gaps, key=lambda x: (x[1] - x[0]).days, reverse=True)
        print(f"\n  Top 10 largest gaps:")
        for i, (start, end) in enumerate(gaps_sorted[:10], 1):
            days = (end - start).days + 1
            print(f"    {i}. {start} to {end} ({days} days)")
    else:
        print("  No gaps - complete data!")
else:
    print("No data to analyze!")

Date Analysis:
  First date: 2022-12-28
  Last date: 2026-04-19
  Total span (days): 1209
  Days with data: 1089
  Missing days: 120
  Coverage: 90.07%

  First missing date: 2022-12-29
  Last missing date: 2026-02-28

Gap Analysis (consecutive missing days):
  Number of gaps: 79
  Largest gap: 7 days

  Top 10 largest gaps:
    1. 2024-12-12 to 2024-12-18 (7 days)
    2. 2024-08-27 to 2024-08-31 (5 days)
    3. 2024-02-06 to 2024-02-09 (4 days)
    4. 2024-03-28 to 2024-03-31 (4 days)
    5. 2025-08-27 to 2025-08-30 (4 days)
    6. 2023-02-04 to 2023-02-06 (3 days)
    7. 2023-04-28 to 2023-04-30 (3 days)
    8. 2023-12-05 to 2023-12-07 (3 days)
    9. 2024-07-29 to 2024-07-31 (3 days)
    10. 2025-03-29 to 2025-03-31 (3 days)


## 7. Generate Summary Report

In [39]:
# Create comprehensive summary report
if len(df_extracted) > 0:
    print("=" * 70)
    print("HYDRO RIVER DATA ANALYSIS SUMMARY")
    print("=" * 70)
    
    print(f"\nTarget Rivers: {', '.join(TARGET_RIVERS)}")
    print(f"Data Range: {first_date.date()} to {last_date.date()}")
    print(f"CSV Files Processed: {len(csv_files)}")
    
    print(f"\nOVERALL STATISTICS:")
    print(f"  Total Records: {len(df_extracted)}")
    print(f"  Total Days in Span: {total_days}")
    print(f"  Days with Data: {present_days}")
    print(f"  Missing Days: {missing_days_count}")
    print(f"  Data Coverage: {coverage_percent:.2f}%")
    
    print(f"\nSTATISTICS BY RIVER:")
    for river in sorted(df_extracted['raul_matched'].unique()):
        df_river = df_extracted[df_extracted['raul_matched'] == river]
        print(f"\n  {river.upper()}:")
        print(f"    Records: {len(df_river)}")
        print(f"    First date: {df_river['date'].min().date()}")
        print(f"    Last date: {df_river['date'].max().date()}")
        
        # Count unique stations
        unique_stations = df_river['statia_hidrometrica'].nunique()
        print(f"    Unique monitoring stations: {unique_stations}")
        
        # Get some sample station names
        sample_stations = df_river['statia_hidrometrica'].unique()[:3]
        print(f"    Sample stations: {', '.join(sample_stations)}")
        
        # Check data completeness for this river
        river_dates = set(df_river['date'].dt.date)
        river_coverage = (len(river_dates) / total_days) * 100
        print(f"    Data coverage: {river_coverage:.2f}%")
    
    print("\n" + "=" * 70)
    print(f"Analysis complete! Total river entries extracted: {len(df_extracted)}")
    print("=" * 70)
else:
    print("ERROR: No data available for analysis!")

HYDRO RIVER DATA ANALYSIS SUMMARY

Target Rivers: Arges, Vedea
Data Range: 2022-12-28 to 2026-04-19
CSV Files Processed: 245

OVERALL STATISTICS:
  Total Records: 2178
  Total Days in Span: 1209
  Days with Data: 1089
  Missing Days: 120
  Data Coverage: 90.07%

STATISTICS BY RIVER:

  ARGES:
    Records: 1089
    First date: 2022-12-28
    Last date: 2026-04-19
    Unique monitoring stations: 1
    Sample stations: Malu Spart
    Data coverage: 90.07%

  VEDEA:
    Records: 1089
    First date: 2022-12-28
    Last date: 2026-04-19
    Unique monitoring stations: 1
    Sample stations: Alexandria
    Data coverage: 90.07%

Analysis complete! Total river entries extracted: 2178


## Export Extracted Data

Export the extracted river data for further analysis or use in other applications.

In [40]:
columns_to_show = [
    'date', 'raul', 'raul_matched', 'statia_hidrometrica', 'CA_cm', 'CI_cm', 'CP_cm', 
    'Qmed_apr_m3_s', 'diag_H_cm', 'diag_Q_m3_s', 'diag_dH_cm', 'diag_GP', 
    'prog_H_cm', 'prog_Q_m3_s', 'prog_dH_cm', 'prog_GP'
]

In [41]:
if len(df_extracted) > 0:
    # Export to CSV
    output_csv = '/home/jovyan/hydro-ocr/extracted_rivers_arges_vedea_dambovita.csv'
    df_extracted[columns_to_show].to_csv(output_csv, index=False)
    print(f"✓ Extracted data saved to: {output_csv}")
    
    # Export summary by river
    for river in sorted(df_extracted['raul_matched'].unique()):
        df_river = df_extracted[df_extracted['raul_matched'] == river]
        output_river_csv = f'/home/jovyan/hydro-ocr/river_{river.lower()}_data.csv'
        df_river[columns_to_show].to_csv(output_river_csv, index=False)
        print(f"✓ {river} data saved to: {output_river_csv}")
else:
    print("No data to export!")

✓ Extracted data saved to: /home/jovyan/hydro-ocr/extracted_rivers_arges_vedea_dambovita.csv
✓ arges data saved to: /home/jovyan/hydro-ocr/river_arges_data.csv
✓ vedea data saved to: /home/jovyan/hydro-ocr/river_vedea_data.csv
